<a href="https://colab.research.google.com/github/jdeepak-4u/my-new-ai-repo/blob/feature-agent/classify_agent_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Problem Statement - Classification Agent with MCP

## Objective

Build an intent classification agent using:

- MCP (Model Context Protocol)
- LangChain
- Azure OpenAI
- Langfuse

The agent should classify user queries into:

- `info`
- `action`
- `summary`

and route requests to the correct MCP tools.

---

## Hackathon Tasks

You need to:

1. Install required packages
2. Configure Azure OpenAI + Langfuse
3. Create MCP tools
4. Start MCP server
5. Connect MCP client
6. Build an intent classifier
7. Evaluate classifier accuracy
8. Build routing agent
9. Improve prompt quality
10. Observe traces in Langfuse

# Step 1 - Install Packages

In [1]:
!pip install -qU   mcp   langchain   langchain-core   langgraph   langchain-mcp-adapters   langchain-groq   langchain-openai   langfuse   pydantic   pandas   scikit-learn   nest_asyncio   python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 482.4/482.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curr

# Step 2 - Import Packages

In [14]:
import os
import getpass

# Step 3 - Configure Environment Variables

Set:

- Groq AI credentials
- Langfuse credentials

In [15]:
def set_secret_if_missing(env_name: str, required: bool = False, default: str | None = None):
    if os.environ.get(env_name):
        print(f"{env_name}: already set")
        return
    if default is not None:
        os.environ[env_name] = default
        print(f"{env_name}: set to default")
        return
    if required:
        os.environ[env_name] = getpass.getpass(f"Enter {env_name}: ")
    else:
        value = getpass.getpass(f"Enter {env_name} or leave blank to skip: ")
        if value:
            os.environ[env_name] = value

# Required for Groq classifier
set_secret_if_missing("GROQ_API_KEY", required=True)

# Optional Langfuse config
set_secret_if_missing("LANGFUSE_PUBLIC_KEY")
set_secret_if_missing("LANGFUSE_SECRET_KEY")
set_secret_if_missing("LANGFUSE_BASE_URL", default="https://cloud.langfuse.com")

print("Configuration cell complete.")


Enter GROQ_API_KEY: ··········
Enter LANGFUSE_PUBLIC_KEY or leave blank to skip: ··········
Enter LANGFUSE_SECRET_KEY or leave blank to skip: ··········
LANGFUSE_BASE_URL: set to default
Configuration cell complete.


## Step 4 Initialize Langfuse tracing

If Langfuse keys are configured, LangChain invocations will emit traces through `CallbackHandler`.

In [16]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

LANGFUSE_ENABLED = bool(os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"))

if LANGFUSE_ENABLED:
    langfuse = get_client()
    langfuse_handler = CallbackHandler()
    print("Langfuse tracing enabled.")
else:
    langfuse = None
    langfuse_handler = None
    print("Langfuse keys not found. Tracing will be skipped, but the notebook will still run.")


def lc_config(run_name: str, tags: list[str] | None = None):
    cfg = {"run_name": run_name, "tags": tags or []}
    if langfuse_handler:
        cfg["callbacks"] = [langfuse_handler]
    return cfg

Langfuse tracing enabled.


# Step 5 - Create MCP Server and register MCP Tools

Create an MCP server using FastMCP.

In [17]:
%%writefile mcp_intent_tools_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("IntentRoutingTools")


@mcp.tool()
def info_tool(query: str) -> str:
    """Use this tool for info intent: answer factual, explanatory, or how-to questions."""
    return (
        "INFO TOOL RESULT\n"
        f"Received query: {query}\n"
        "Suggested handling: provide facts, definitions, explanation, or guidance."
    )


@mcp.tool()
def action_tool(query: str) -> str:
    """Use this tool for action intent: perform, create, update, book, send, schedule, or execute something."""
    return (
        "ACTION TOOL RESULT\n"
        f"Received request: {query}\n"
        "Suggested handling: execute or prepare the requested operation after validation."
    )


@mcp.tool()
def summary_tool(text: str) -> str:
    """Use this tool for summary intent: summarize, condense, extract key points, or produce a recap."""
    clean = " ".join(text.split())
    words = clean.split()
    preview = " ".join(words[:35]) + ("..." if len(words) > 35 else "")
    return (
        "SUMMARY TOOL RESULT\n"
        f"Input word count: {len(words)}\n"
        f"Concise preview: {preview}"
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")


Writing mcp_intent_tools_server.py


# Step 6 Connect MCP client and load tools

The MCP stdio client starts `mcp_intent_tools_server.py` as a subprocess, initializes the MCP session, and loads the server tools as LangChain tools.

In [19]:
import asyncio
import nest_asyncio
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

nest_asyncio.apply()

class MCPToolClient:
    def __init__(self, server_script: str):
        self.server_script = server_script
        self.stack = AsyncExitStack()
        self.session = None
        self.tools = []
        self.tool_by_name = {}

    async def connect(self):
        server_params = StdioServerParameters(
            command="python",
            args=[self.server_script],
        )
        read, write = await self.stack.enter_async_context(stdio_client(server_params))
        self.session = await self.stack.enter_async_context(ClientSession(read, write))
        await self.session.initialize()
        self.tools = await load_mcp_tools(self.session)
        self.tool_by_name = {tool.name: tool for tool in self.tools}
        return self.tools

    async def close(self):
        await self.stack.aclose()

mcp_client = MCPToolClient("mcp_intent_tools_server.py")
tools = asyncio.get_event_loop().run_until_complete(mcp_client.connect())
print("Loaded MCP tools:", [tool.name for tool in tools])


UnsupportedOperation: fileno